In [ ]:
import sys
sys.path.append('/opt/workspace')

from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, concat_ws, sha2, to_json, struct, current_timestamp,
)
from config.settings import oracle_config, clickhouse_config, spark_config, app_config

In [ ]:
ref_date = datetime.now().strftime("%Y-%m-%d")
ref_date


In [ ]:
spark = (
    SparkSession.builder
    .appName(f"BronzeIngestion_{ref_date}")
    .config("spark.driver.memory", spark_config.driver_memory)
    .config("spark.executor.memory", spark_config.executor_memory)
    .config("spark.executor.cores", spark_config.executor_cores)
    .config("spark.sql.shuffle.partitions", spark_config.sql_shuffle_partitions)
    .config("spark.sql.adaptive.enabled", spark_config.sql_adaptive_enabled)
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.jars", "/opt/spark/jars/ojdbc11.jar,/opt/spark/jars/clickhouse-jdbc.jar")
    .getOrCreate()
)


In [ ]:
oracle_jdbc_url = f"jdbc:oracle:thin:@{oracle_config.host}:{oracle_config.port}/{oracle_config.service}"
oracle_jdbc_url


In [ ]:
schema_name = "ADMBI_PRD"
table_name = "YOUR_TABLE_NAME"
primary_keys = ["ID"]


In [ ]:
df_oracle = (
    spark.read
    .format("jdbc")
    .option("url", oracle_jdbc_url)
    .option("dbtable", f"{schema_name}.{table_name}")
    .option("user", oracle_config.user)
    .option("password", oracle_config.password)
    .option("driver", "oracle.jdbc.driver.OracleDriver")
    .option("fetchsize", app_config.batch_size)
    .option("numPartitions", "10")
    .load()
)


In [ ]:
df_oracle.printSchema()


In [ ]:
primary_key_str = concat_ws("|||", *[col(pk) for pk in primary_keys])
all_columns = [col(c) for c in df_oracle.columns]
row_data = to_json(struct(*all_columns))
row_hash_input = concat_ws("|||", *all_columns)


In [ ]:
df_bronze = df_oracle.select(
    lit(ref_date).cast("date").alias("ref_date"),
    lit(table_name).alias("table_name"),
    primary_key_str.alias("primary_key"),
    sha2(row_hash_input, 256).alias("row_hash"),
    row_data.alias("data"),
    current_timestamp().alias("ingestion_timestamp"),
)


In [ ]:
df_bronze.show(5, truncate=False)


In [ ]:
protocol = "https" if clickhouse_config.secure else "http"
clickhouse_jdbc_url = (
    f"jdbc:clickhouse://{protocol}://{clickhouse_config.host}:"
    f"{clickhouse_config.port}/{clickhouse_config.database}"
)
clickhouse_jdbc_url


In [ ]:
df_bronze.write \
    .format("jdbc") \
    .option("url", clickhouse_jdbc_url) \
    .option("dbtable", "bronze.snapshot_raw") \
    .option("user", clickhouse_config.user) \
    .option("password", clickhouse_config.password) \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("batchsize", app_config.batch_size) \
    .mode("append") \
    .save()


In [ ]:
spark.stop()
